# Laboratorio: técnicas de *chunking* (fragmentación) para RAG
**Curso: Tecnologías Emergentes**

En el chatbot RAG que construimos (`app_llm.ipynb`) el PDF se dividía así:

```python
trozos = [texto[i:i+1000] for i in range(0, len(texto), 1000)]
```

Es la forma más simple de fragmentar un documento, pero **no la única ni la mejor**. En este laboratorio compararemos **7 técnicas de fragmentación** sobre el mismo texto, con el **mismo modelo de embeddings** (`gemini-embedding-001`) y la **misma búsqueda vectorial** (MongoDB Atlas `$vectorSearch`), y mediremos cuál recupera mejor la información.

**Objetivos**
1. Entender qué es el *chunking* y por qué influye en la calidad de un sistema RAG.
2. Implementar 7 técnicas de fragmentación en Python y con LangChain.
3. Medir cada técnica con métricas objetivas: **Hit@3** y **MRR**.
4. Decidir qué técnica aplicar según el tipo de documento.

**Mapa del laboratorio**

| Sección | Contenido |
|---|---|
| 0 | Teoría: ¿qué es fragmentar un documento? |
| 1–2 | Instalación y configuración (secretos, Gemini, MongoDB Atlas) |
| 3 | Corpus y preguntas de evaluación |
| 4 | Métricas: Hit@3 y MRR explicadas con ejemplos |
| 5 | Funciones de apoyo |
| 6–12 | Las 7 técnicas de fragmentación |
| 13–14 | Comparación final y RAG lado a lado |
| 15 | Resumen: ¿qué técnica usar según el caso? |
| 16 | Limpieza |

## 0. Teoría: ¿qué es fragmentar un documento?

**Chunking** (fragmentación) es dividir un documento largo en piezas más pequeñas llamadas **fragmentos** (*chunks*). Cada fragmento se convierte en un embedding y se guarda en la base vectorial. Cuando el usuario pregunta, se recuperan los fragmentos más parecidos y se envían al LLM como contexto.

```
Documento ──► Fragmentos ──► Embeddings ──► Índice vectorial
                                                  │
Pregunta ──► Embedding de la pregunta ──► Búsqueda (top k) ──► LLM ──► Respuesta
```

**¿Por qué no enviar el documento completo?**
- Los modelos tienen un **límite de tokens** de entrada (el de embeddings `gemini-embedding-001` acepta hasta 2,048 tokens por texto).
- Un embedding de un documento entero "promedia" muchos temas y pierde **precisión**.
- Enviar todo al LLM en cada pregunta es **más caro y más lento**.

**El dilema del tamaño**

| Fragmentos pequeños | Fragmentos grandes |
|---|---|
| ✅ Embedding preciso (un solo tema) | ✅ Conservan el contexto |
| ❌ Pueden perder el contexto (¿de qué habla "este sistema"?) | ❌ El embedding mezcla temas y se "diluye" |
| ❌ Más fragmentos, más llamadas al modelo | ❌ Más ruido para el LLM |

**Parámetros que veremos en casi todas las técnicas**
- `chunk_size` (tamaño): longitud máxima del fragmento, en caracteres, tokens u oraciones.
- `chunk_overlap` (solapamiento): cuánto texto comparten dos fragmentos consecutivos.

> **Analogía:** fragmentar es como dividir un libro en fichas de estudio. Si cortas cada ficha cada 20 líneas exactas, algunas ideas quedarán partidas entre dos fichas; si haces una ficha por tema, cada una tendrá sentido por sí sola.

## 1. Instalación de librerías
- `pymongo`: cliente de MongoDB Atlas.
- `google-genai`: SDK de Gemini (embeddings y generación).
- `langchain-text-splitters`: fragmentadores de LangChain.
- `tiktoken`: tokenizador usado por el fragmentador por tokens.

In [ ]:
!pip -q install pymongo google-genai langchain-text-splitters tiktoken

## 2. Configuración

### 2.1 Secretos de Colab
Las credenciales **nunca** se escriben en el código. En el panel izquierdo de Colab (ícono 🔑 **Secrets**) registre:

| Secreto | Valor |
|---|---|
| `GOOGLE_API_KEY` | API key de Google AI Studio |
| `MONGODB_URI` | `mongodb+srv://<usuario>:<password>@<cluster>.mongodb.net/` |

Active **Notebook access** en ambos.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
MONGODB_URI    = userdata.get("MONGODB_URI")
print("Secretos cargados ✔")

### 2.2 Clientes de Gemini y MongoDB Atlas
Usaremos una base de datos y colección **nuevas** para no mezclar estos fragmentos con los del chatbot:
- Base de datos: `chunking_lab_db`
- Colección: `fragmentos`

Cada documento guardará el texto, su embedding y el campo **`tecnica`**, que indica con qué técnica fue fragmentado.

In [ ]:
import time
import pymongo
from google import genai
from google.genai import types

client_genai = genai.Client(api_key=GOOGLE_API_KEY)
client_mongo = pymongo.MongoClient(MONGODB_URI)
collection   = client_mongo["chunking_lab_db"]["fragmentos"]

MODELO_EMBEDDING = "gemini-embedding-001"   # 3072 dimensiones
MODELO_LLM       = "gemini-3.1-flash-lite"   # gemini-2.5-flash ya no está disponible para cuentas nuevas
INDICE           = "vector_index_lab"

print("Conexión OK. Documentos en la colección:", collection.count_documents({}))

### 2.3 Índice vectorial con filtro
El índice es igual al del chatbot, con una diferencia: agregamos `tecnica` como **campo de filtro**. Así, `$vectorSearch` puede buscar **solo entre los fragmentos de una técnica**, y comparamos todas las técnicas en una sola colección.

| Campo | Tipo | Para qué |
|---|---|---|
| `embedding` | `vector` (3072, `dotProduct`) | Búsqueda por similitud |
| `tecnica` | `filter` | Restringir la búsqueda a una técnica |

> `dotProduct` funciona bien aquí porque los vectores de 3072 dimensiones de `gemini-embedding-001` ya vienen normalizados (longitud 1), y en ese caso el producto punto equivale a la similitud coseno.

In [ ]:
from pymongo.operations import SearchIndexModel

if collection.count_documents({}) == 0:          # la colección debe existir para crear el índice
    collection.insert_one({"_init": True})

modelo_indice = SearchIndexModel(
    definition={
        "fields": [
            {"type": "vector", "path": "embedding", "numDimensions": 3072, "similarity": "dotProduct"},
            {"type": "filter", "path": "tecnica"},
        ]
    },
    name=INDICE,
    type="vectorSearch",
)
try:
    collection.create_search_index(model=modelo_indice)
    print("Índice creado; esperando a que esté listo...")
except Exception as e:
    print("Aviso (puede que ya exista):", e)

# Esperar a que el índice esté listo para consultas
while True:
    info = list(collection.list_search_indexes(INDICE))
    if info and info[0].get("queryable"):
        print("Índice READY ✔")
        break
    time.sleep(5)

collection.delete_many({"_init": True})

## 3. Corpus y preguntas de evaluación

### 3.1 Corpus
Usaremos un documento corto en español (~6,000 caracteres) sobre la adopción de la nube en una empresa **ficticia**. Tiene títulos y secciones, igual que muchos documentos reales (manuales, reglamentos, informes). Al ser corto, podemos **leer los fragmentos** y ver qué hace cada técnica.

In [ ]:
CORPUS = """# Guía de adopción de la nube de Andina Retail

## 1. Contexto de la empresa

Andina Retail es una cadena ficticia de supermercados con presencia en Perú y Chile. Cuenta con 148 tiendas físicas, un canal de comercio electrónico y una aplicación móvil que concentra el 22 % de las ventas totales. Sus sistemas actuales se ejecutan en un centro de datos propio ubicado en Lima, cuyo contrato de arrendamiento vence en diciembre de 2027. La gerencia de tecnología decidió migrar a la nube para ganar elasticidad en campañas como el Cyber Day, cuando el tráfico de la tienda en línea se multiplica por seis. El proyecto tiene un presupuesto aprobado de 2.4 millones de dólares y será liderado por la Oficina de Transformación Digital.

## 2. Estrategia de migración

Cada aplicación fue evaluada con el modelo de las 6 R: retirar, retener, rehospedar, replataformar, recomprar y refactorizar. El sistema de punto de venta se rehospedará sin cambios en máquinas virtuales durante la primera ola. El sistema de planillas se reemplazará por un software como servicio, es decir, se aplicará la estrategia de recompra. El motor de precios, que hoy es un monolito difícil de escalar, será refactorizado en microservicios con contenedores. Las aplicaciones de reportes que nadie usa desde 2023 serán retiradas. La migración se ejecutará en tres olas de cuatro meses cada una, empezando por los sistemas de menor riesgo. Antes de cada ola habrá una prueba piloto en una tienda de Arequipa; si los indicadores son estables, se seguirá con el resto. Las aplicaciones que dependan de hardware especial, como las balanzas, se mantendrán en las tiendas y se conectarán a la nube por una red privada.

## 3. Seguridad y cumplimiento

La seguridad se basa en el modelo de responsabilidad compartida: el proveedor protege la infraestructura física y la empresa protege sus datos, identidades y configuraciones. Todo acceso administrativo exige autenticación multifactor. Las credenciales de servicio se rotarán automáticamente cada 90 días y se almacenarán en un gestor de secretos, nunca en el código fuente. Los datos personales de clientes se cifrarán en reposo y en tránsito para cumplir con la Ley de Protección de Datos Personales. Además, se activará un registro centralizado de auditoría que conservará los eventos durante un año.

## 4. Arquitectura de datos

Los datos de ventas, inventario y clientes hoy están dispersos en once bases de datos distintas, lo que dificulta obtener una visión única del negocio. En la nube se construirá un lago de datos con tres capas: bronce para los datos crudos, plata para los datos limpios y oro para los indicadores listos para el negocio. La información de las tiendas se cargará cada quince minutos mediante procesos de ingesta automatizados. Sobre la capa oro se publicarán tableros de autoservicio para los gerentes de tienda. Los equipos de ciencia de datos usarán esta misma plataforma para entrenar modelos de pronóstico de demanda, que permitirán reducir los quiebres de stock en las categorías de frescos. Cada conjunto de datos tendrá un responsable de negocio y un diccionario que describa sus campos.

## 5. Gestión de costos

La empresa adoptará prácticas FinOps para que cada área conozca y controle su gasto en la nube. Todos los recursos llevarán etiquetas obligatorias de centro de costo, ambiente y responsable. Para las cargas estables, como las bases de datos de inventario, se comprarán instancias reservadas a tres años, con un ahorro estimado de 38 % frente al precio por demanda. Los ambientes de desarrollo se apagarán automáticamente fuera del horario laboral. Se configurarán alertas de presupuesto que avisarán al responsable cuando el gasto de su área supere el 80 % de lo planificado. Cada mes, el comité de costos revisará un tablero con el gasto por área y las desviaciones frente al presupuesto.

## 6. Plan de capacitación

Ninguna migración funciona sin personas preparadas. Por ello, cada miembro del equipo de tecnología recibirá 40 horas de capacitación en fundamentos de nube, seguridad y automatización. Los arquitectos deberán obtener una certificación profesional antes de la segunda ola. También se creará un centro de excelencia en la nube, integrado por seis especialistas, que definirá estándares, plantillas reutilizables y buenas prácticas para el resto de la organización. Los colaboradores de tienda no serán ajenos al cambio: recibirán talleres cortos sobre las nuevas herramientas de reportes y sobre cómo reportar incidentes desde su celular.

## 7. Gestión de riesgos

El comité de proyecto identificó los riesgos principales y definió una respuesta para cada uno. El primer riesgo es la interrupción de las ventas durante el traslado de los sistemas críticos; para mitigarlo, las migraciones se realizarán los martes de madrugada, que es la franja con menor número de transacciones, y siempre con un plan de reversión probado. El segundo riesgo es la dependencia de un solo proveedor, por lo que los nuevos servicios se construirán con contenedores y herramientas de código abierto que faciliten un cambio futuro. El tercer riesgo es la falta de personal especializado, que se aborda con el plan de capacitación y con la contratación temporal de un socio consultor. Todos los riesgos se revisarán cada dos semanas en el comité.

## 8. Indicadores de éxito

El avance del proyecto se medirá con indicadores concretos. La meta de disponibilidad de la tienda en línea es de 99.9 % mensual. El tiempo de despliegue de una nueva versión debe bajar de dos semanas a un día. El costo total de operación debe reducirse en 20 % al finalizar el segundo año. Finalmente, la satisfacción de los equipos internos se evaluará con una encuesta trimestral. Si dos indicadores consecutivos no alcanzan su meta, el comité de proyecto deberá presentar un plan de corrección en un plazo máximo de diez días."""

print(f"Caracteres: {len(CORPUS):,} | Palabras: {len(CORPUS.split()):,}")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(CORPUS))

### 3.2 Preguntas de evaluación
Para comparar técnicas de forma **objetiva**, definimos preguntas cuya respuesta conocemos. Para cada una indicamos una **frase clave**: el fragmento recuperado es **relevante** si contiene esa frase completa.

> Fíjate en que las preguntas **no repiten** las palabras exactas del texto (p. ej. "renovar las claves" vs. "rotarán las credenciales"): la búsqueda debe ser **semántica**, no por palabras.

In [ ]:
PREGUNTAS = [
    {"pregunta": "¿Cuántos locales físicos tiene la empresa?",                       "frase_clave": "148 tiendas"},
    {"pregunta": "¿Qué se hará con el sistema de planillas?",                        "frase_clave": "estrategia de recompra"},
    {"pregunta": "¿Cada cuánto tiempo se deben renovar las claves de los servicios?", "frase_clave": "cada 90 días"},
    {"pregunta": "¿Cuánto dinero se ahorra con las instancias reservadas?",          "frase_clave": "ahorro estimado de 38 %"},
    {"pregunta": "¿Cuánta formación recibirá el personal de TI?",                    "frase_clave": "40 horas de capacitación"},
    {"pregunta": "¿Cuándo se migrarán los sistemas críticos para no afectar las ventas?", "frase_clave": "los martes de madrugada"},
    {"pregunta": "¿Con qué frecuencia llegan los datos de las tiendas al lago de datos?", "frase_clave": "cada quince minutos"},
]

import pandas as pd
pd.DataFrame(PREGUNTAS)

## 4. Métricas de evaluación: Hit@3 y MRR

Para cada pregunta, la búsqueda vectorial devuelve los **3 fragmentos más parecidos**, ordenados del más al menos similar (posiciones 1, 2 y 3). Lo que nos interesa es **en qué posición aparece el primer fragmento relevante** (el que contiene la frase clave).

### 4.1 Hit@3 (tasa de acierto en el top 3)
> ¿El fragmento correcto apareció **entre los 3 primeros**? Sí = 1, No = 0.

**Hit@3** es el promedio sobre todas las preguntas, es decir, el **porcentaje de preguntas** que encontraron su respuesta en el top 3.

### 4.2 MRR (*Mean Reciprocal Rank*, rango recíproco medio)
Hit@3 no distingue entre encontrar la respuesta en la posición 1 o en la 3. MRR **premia que el fragmento correcto aparezca primero**:

$$\text{RR} = \frac{1}{\text{posición del primer fragmento relevante}} \qquad \text{MRR} = \text{promedio de RR}$$

| Posición del fragmento relevante | 1 | 2 | 3 | No aparece |
|---|---|---|---|---|
| **RR** | 1.00 | 0.50 | 0.33 | 0 |

### 4.3 Ejemplo
Supongamos 4 preguntas con estos resultados:

| Pregunta | Posición del relevante | Hit@3 | RR |
|---|---|---|---|
| P1 | 1 | 1 | 1.00 |
| P2 | 3 | 1 | 0.33 |
| P3 | No aparece | 0 | 0.00 |
| P4 | 2 | 1 | 0.50 |
| **Promedio** | | **0.75** | **0.46** |

**Interpretación:**
- **Hit@3 = 0.75** → en 3 de cada 4 preguntas, el LLM recibiría el fragmento con la respuesta.
- **MRR = 0.46** → en promedio, el fragmento correcto aparece entre la 2.ª y la 3.ª posición. Un MRR cercano a 1 indica que casi siempre aparece **primero**.

> **Importante:** si el fragmento correcto no llega al LLM, el LLM **no puede** responder bien (o "alucina"). Por eso medimos la recuperación **antes** de la generación.

In [ ]:
def hit_y_rr(posicion, k=3):
    # posicion: 1, 2, 3... o None si el fragmento relevante no apareció
    if posicion is None or posicion > k:
        return 0, 0.0
    return 1, 1 / posicion

ejemplo = {"P1": 1, "P2": 3, "P3": None, "P4": 2}
filas = [(p, pos, *hit_y_rr(pos)) for p, pos in ejemplo.items()]
df_ej = pd.DataFrame(filas, columns=["pregunta", "posicion", "hit@3", "rr"])
display(df_ej)
print(f"Hit@3 = {df_ej['hit@3'].mean():.2f} | MRR = {df_ej['rr'].mean():.2f}")

## 5. Funciones de apoyo
Estas funciones se usan con **todas** las técnicas, para que la comparación sea justa:

| Función | Qué hace |
|---|---|
| `crear_embeddings(textos, task_type)` | Genera embeddings en **lotes** (menos llamadas a la API) y reintenta si se supera el límite gratuito |
| `ingestar(tecnica, fragmentos)` | Borra los fragmentos anteriores de esa técnica, genera los embeddings y los inserta en Atlas |
| `buscar(pregunta, tecnica, k)` | Ejecuta `$vectorSearch` filtrando por técnica |
| `evaluar(tecnica)` | Calcula la posición del fragmento relevante, Hit@3 y MRR para cada pregunta |
| `mostrar_fragmentos(fragmentos)` | Muestra los fragmentos con colores para ver **dónde corta** cada técnica |
| `estadisticas(tecnica, fragmentos)` | Cantidad, longitud, % que termina en fin de oración y frases clave partidas |

In [ ]:
import re
import numpy as np
from html import escape
from IPython.display import HTML

# ---------- Embeddings ----------
def crear_embeddings(textos, task_type="RETRIEVAL_DOCUMENT", lote=50):
    vectores = []
    for i in range(0, len(textos), lote):
        bloque = textos[i:i + lote]
        for intento in range(5):
            try:
                resp = client_genai.models.embed_content(
                    model=MODELO_EMBEDDING,
                    contents=bloque,
                    config=types.EmbedContentConfig(task_type=task_type),
                )
                vectores += [e.values for e in resp.embeddings]
                break
            except Exception as e:
                if any(x in str(e) for x in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")):
                    print("Límite de la capa gratuita o Gemini saturado; esperando 30 s...")
                    time.sleep(30)
                else:
                    raise
    return vectores

# Embeddings de las preguntas: se calculan una sola vez y se reutilizan
_cache_preguntas = {}
def embedding_pregunta(pregunta):
    if pregunta not in _cache_preguntas:
        _cache_preguntas[pregunta] = crear_embeddings([pregunta], task_type="RETRIEVAL_QUERY")[0]
    return _cache_preguntas[pregunta]

# ---------- Ingesta y búsqueda ----------
def ingestar(tecnica, fragmentos, metadatos=None):
    collection.delete_many({"tecnica": tecnica})
    vectores = crear_embeddings(fragmentos)
    docs = []
    for i, (texto, emb) in enumerate(zip(fragmentos, vectores)):
        doc = {"tecnica": tecnica, "orden": i, "texto": texto, "embedding": emb}
        if metadatos:
            doc.update(metadatos[i])
        docs.append(doc)
    collection.insert_many(docs)
    # Atlas indexa los nuevos vectores en unos segundos: esperar a que la búsqueda filtrada los devuelva
    esperado = min(3, len(docs))
    for _ in range(30):
        n = len(list(collection.aggregate([{"$vectorSearch": {"index": INDICE, "path": "embedding",
            "queryVector": vectores[0], "numCandidates": 100, "limit": 3, "filter": {"tecnica": tecnica}}}])))
        if n >= esperado:
            break
        time.sleep(5)
    print(f"[{tecnica}] {len(docs)} fragmentos insertados")

def buscar(pregunta, tecnica, k=3):
    pipeline = [
        {"$vectorSearch": {
            "index": INDICE, "path": "embedding",
            "queryVector": embedding_pregunta(pregunta),
            "numCandidates": 100, "limit": k,
            "filter": {"tecnica": tecnica},
        }},
        {"$project": {"_id": 0, "orden": 1, "texto": 1, "score": {"$meta": "vectorSearchScore"}}},
    ]
    return list(collection.aggregate(pipeline))

# ---------- Evaluación ----------
def normalizar(t):
    return re.sub(r"\s+", " ", t).strip().lower()

resultados = {}   # aquí se acumulan las métricas de cada técnica

def evaluar(tecnica, k=3):
    filas = []
    for p in PREGUNTAS:
        recuperados = buscar(p["pregunta"], tecnica, k)
        posicion = next((i + 1 for i, r in enumerate(recuperados)
                         if normalizar(p["frase_clave"]) in normalizar(r["texto"])), None)
        hit, rr = hit_y_rr(posicion, k)
        filas.append({"pregunta": p["pregunta"], "posicion": posicion, "hit@3": hit, "rr": round(rr, 2)})
    df = pd.DataFrame(filas)
    resultados.setdefault(tecnica, {}).update({"Hit@3": df["hit@3"].mean(), "MRR": df["rr"].mean()})
    display(df)
    print(f"▶ {tecnica}:  Hit@3 = {df['hit@3'].mean():.2f}  |  MRR = {df['rr'].mean():.2f}")

# ---------- Visualización y estadísticas ----------
COLORES = ["#DCEBFA", "#FCE4D6", "#E2F0D9", "#FFF2CC", "#EADCF4", "#D9F2F2"]

def mostrar_fragmentos(fragmentos, max_frag=6, resaltar_inicio=0):
    # resaltar_inicio: n.º de caracteres iniciales que se resaltan en amarillo (para ver el solapamiento)
    html = ""
    for i, f in enumerate(fragmentos[:max_frag]):
        n = resaltar_inicio if i > 0 else 0
        inicio = f"<mark style='background:#FFD966'>{escape(f[:n])}</mark>" if n else ""
        html += (f"<div style='background:{COLORES[i % len(COLORES)]};padding:6px 10px;margin:4px 0;"
                 f"border-radius:6px;font-family:monospace;font-size:12px;white-space:pre-wrap;color:#000'>"
                 f"<b>[{i}] ({len(f)} car.)</b> {inicio}{escape(f[n:])}<b style='color:#C00000'>⏎</b></div>")
    if len(fragmentos) > max_frag:
        html += f"<i>... y {len(fragmentos) - max_frag} fragmentos más</i>"
    display(HTML(html))

def termina_en_oracion(f):
    f = f.strip()
    return f.endswith((".", "!", "?", ":")) or f.startswith("#")

def estadisticas(tecnica, fragmentos):
    long = [len(f) for f in fragmentos]
    cortadas = [p["frase_clave"] for p in PREGUNTAS
                if not any(normalizar(p["frase_clave"]) in normalizar(f) for f in fragmentos)]
    stats = {
        "fragmentos": len(fragmentos),
        "long. media": round(np.mean(long)),
        "long. mín": min(long),
        "long. máx": max(long),
        "caracteres indexados": sum(long),
        "% termina en oración": round(100 * np.mean([termina_en_oracion(f) for f in fragmentos])),
        "frases clave partidas": len(cortadas),
    }
    resultados.setdefault(tecnica, {}).update(stats)
    display(pd.DataFrame([stats], index=[tecnica]))
    if cortadas:
        print("⚠️ Frases clave partidas entre dos fragmentos (nunca podrán recuperarse completas):", cortadas)

## 6. Técnica 1 — Tamaño fijo por caracteres

**Idea:** cortar el texto cada *N* caracteres, sin importar dónde caiga el corte. Es lo que hace `app_llm.ipynb` con `N = 1000`.

> **Analogía:** cortar un periódico en tiras de 10 cm con una guillotina, sin mirar el texto.

| Parámetro | Valor en este lab |
|---|---|
| `tamano` | 500 caracteres (usamos 500 porque el corpus es corto) |
| `solapamiento` | 0 |

**Lo que debes observar:** fragmentos que empiezan o terminan **a mitad de una palabra u oración**, y frases clave que quedan **partidas** entre dos fragmentos.

In [ ]:
def fragmentar_fijo(texto, tamano=500):
    return [texto[i:i + tamano] for i in range(0, len(texto), tamano)]

fragmentos = fragmentar_fijo(CORPUS, 500)
mostrar_fragmentos(fragmentos)
estadisticas("1. Fijo (caracteres)", fragmentos)

In [ ]:
ingestar("1. Fijo (caracteres)", fragmentos)
evaluar("1. Fijo (caracteres)")

| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Muy simple y rápida | Corta palabras, oraciones e ideas |
| Tamaño predecible (costo predecible) | La información de los bordes se pierde o queda incompleta |
| No depende del idioma | Ignora la estructura del documento |

## 7. Técnica 2 — Tamaño fijo con solapamiento (*overlap*)

**Idea:** igual que la anterior, pero cada fragmento **repite los últimos caracteres** del fragmento anterior. Si una frase queda cortada en el borde, aparecerá completa en alguno de los dos fragmentos.

> **Analogía:** al fotocopiar un libro página por página, repetir las dos últimas líneas de la hoja anterior para no perder el hilo.

```
Sin solapamiento: [0────500)[500────1000)[1000────1500)
Con solapamiento: [0────500)
                        [400────900)
                              [800────1300)     paso = tamano − solapamiento
```

| Parámetro | Valor |
|---|---|
| `tamano` | 500 |
| `solapamiento` | 100 (20 %) → paso de 400 |

**Lo que debes observar:** el texto resaltado en amarillo es el que se repite del fragmento anterior. Compara las **frases clave partidas** y los **caracteres indexados** con la técnica 1.

In [ ]:
def fragmentar_fijo_solapado(texto, tamano=500, solapamiento=100):
    paso = tamano - solapamiento
    return [texto[i:i + tamano] for i in range(0, len(texto), paso) if texto[i:i + tamano].strip()]

fragmentos = fragmentar_fijo_solapado(CORPUS, 500, 100)
mostrar_fragmentos(fragmentos, resaltar_inicio=100)
estadisticas("2. Fijo + solapamiento", fragmentos)

In [ ]:
ingestar("2. Fijo + solapamiento", fragmentos)
evaluar("2. Fijo + solapamiento")

| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Reduce la pérdida de información en los bordes | Más fragmentos → más embeddings, más almacenamiento y más costo |
| Sigue siendo simple | Sigue cortando palabras y oraciones |
| Mejora la continuidad del contexto | Puede devolver fragmentos casi duplicados en el top k |

> Regla práctica habitual: solapamiento entre **10 % y 20 %** del tamaño del fragmento.

## 8. Técnica 3 — Por tokens

**Idea:** los modelos no leen caracteres sino **tokens** (trozos de palabra). Los límites de los modelos (y su precio) se expresan en tokens, así que tiene sentido fragmentar por tokens.

```
"capacitación"  →  ["cap", "ac", "it", "ación"]   (ejemplo ilustrativo)
```

- En inglés, 1 token ≈ 4 caracteres; en español suele haber **más tokens por palabra** (acentos, palabras largas).
- Cada familia de modelos tiene su **propio tokenizador**. Aquí usamos `TokenTextSplitter` de LangChain con el tokenizador `cl100k_base` (de `tiktoken`) y luego verificamos el conteo con el tokenizador **de Gemini** (`count_tokens`).

| Parámetro | Valor |
|---|---|
| `chunk_size` | 120 tokens |
| `chunk_overlap` | 20 tokens |

In [ ]:
from langchain_text_splitters import TokenTextSplitter

splitter_tokens = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=120, chunk_overlap=20)
fragmentos = splitter_tokens.split_text(CORPUS)

mostrar_fragmentos(fragmentos)
estadisticas("3. Por tokens", fragmentos)

**Caracteres vs. tokens:** comparemos, para los primeros fragmentos, los tokens según `tiktoken` y según el tokenizador de Gemini. Los conteos **no coinciden**: por eso, al fragmentar por tokens, conviene usar el tokenizador del modelo que se va a utilizar o dejar un margen de seguridad.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

filas = []
for f in fragmentos[:3]:
    gemini_tokens = client_genai.models.count_tokens(model=MODELO_LLM, contents=f).total_tokens
    filas.append({"caracteres": len(f), "tokens tiktoken": len(enc.encode(f)), "tokens Gemini": gemini_tokens,
                  "caracteres por token (Gemini)": round(len(f) / gemini_tokens, 1)})
pd.DataFrame(filas)

In [ ]:
ingestar("3. Por tokens", fragmentos)
evaluar("3. Por tokens")

| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Controla con precisión el límite del modelo y el costo | Depende del tokenizador (cada modelo cuenta distinto) |
| Útil cuando los fragmentos se acercan al límite de tokens | Sigue sin respetar oraciones ni secciones |

## 9. Técnica 4 — Por oraciones

**Idea:** primero dividir el texto en **oraciones** y luego agrupar *N* oraciones por fragmento. Así ningún fragmento corta una oración a la mitad.

> **Analogía:** armar fichas de estudio copiando frases completas, de 4 en 4.

Para detectar oraciones usamos una **expresión regular**: un corte después de `.`, `!` o `?` seguido de un espacio. Los títulos (`##`) se conservan como una unidad.

| Parámetro | Valor |
|---|---|
| `n_oraciones` | 4 por fragmento |
| `solapamiento` | 1 oración |

> Limitación: la regex se confunde con abreviaturas (`Sr.`, `p. ej.`) o decimales (`2.4`). Librerías como **spaCy** o **NLTK** detectan oraciones con más precisión.

In [ ]:
def dividir_oraciones(texto):
    oraciones = []
    for linea in texto.split("\n"):
        linea = linea.strip()
        if not linea:
            continue
        if linea.startswith("#"):              # un título se conserva como una "oración"
            oraciones.append(linea)
        else:
            oraciones += [o for o in re.split(r"(?<=[.!?])\s+", linea) if o.strip()]
    return oraciones

def fragmentar_por_oraciones(texto, n_oraciones=4, solapamiento=1):
    oraciones = dividir_oraciones(texto)
    paso = n_oraciones - solapamiento
    return [" ".join(oraciones[i:i + n_oraciones]) for i in range(0, len(oraciones), paso)
            if i == 0 or i + solapamiento < len(oraciones)]

oraciones = dividir_oraciones(CORPUS)
print(f"Se detectaron {len(oraciones)} oraciones. Ejemplos:")
for o in oraciones[2:5]:
    print("  •", o)

fragmentos = fragmentar_por_oraciones(CORPUS, 4, 1)
mostrar_fragmentos(fragmentos)
estadisticas("4. Por oraciones", fragmentos)

In [ ]:
ingestar("4. Por oraciones", fragmentos)
evaluar("4. Por oraciones")

| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Nunca corta una oración | El tamaño varía según la longitud de las oraciones |
| Fragmentos legibles y con sentido | Un fragmento puede mezclar el final de una sección con el inicio de otra |
| Simple de implementar | Depende de una buena detección de oraciones (idioma, abreviaturas) |

## 10. Técnica 5 — Recursiva (`RecursiveCharacterTextSplitter`)

**Idea:** intentar cortar por los separadores **más "naturales" primero** y solo bajar a separadores más finos si el fragmento sigue siendo demasiado grande:

```
1. "\n\n"  (párrafos)
2. "\n"    (líneas)
3. ". "    (oraciones)
4. " "     (palabras)
5. ""      (caracteres, último recurso)
```

**Algoritmo (simplificado):**
1. Divide el texto con el primer separador.
2. Une piezas consecutivas mientras no superen `chunk_size`.
3. Si una pieza sola supera `chunk_size`, vuelve a dividirla con el **siguiente** separador (de ahí lo de *recursiva*).

> **Analogía:** para repartir un libro en cajas, primero intentas meter capítulos completos; si un capítulo no entra, lo separas por párrafos; si un párrafo no entra, por oraciones.

Es la técnica **más usada por defecto** en LangChain y un muy buen punto de partida.

| Parámetro | Valor |
|---|---|
| `chunk_size` | 500 caracteres |
| `chunk_overlap` | 50 caracteres |
| `separators` | `["\n\n", "\n", ". ", " ", ""]` (agregamos `". "` a los separadores por defecto) |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter_rec = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    keep_separator="end",     # el punto se queda al final de la oración
)
fragmentos = splitter_rec.split_text(CORPUS)
mostrar_fragmentos(fragmentos)
estadisticas("5. Recursiva", fragmentos)

In [ ]:
ingestar("5. Recursiva", fragmentos)
evaluar("5. Recursiva")

| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Respeta párrafos y oraciones siempre que puede | Los tamaños son irregulares |
| Funciona bien con casi cualquier texto | No "entiende" el significado; solo usa separadores |
| Configurable y disponible en LangChain | Requiere ajustar `chunk_size` y separadores según el tipo de documento |

## 11. Técnica 6 — Estructural (por títulos y secciones)

**Idea:** usar la **estructura del documento** (títulos, secciones, páginas, artículos) como límite de los fragmentos. Cada fragmento es una sección completa y guarda su título como **metadato**.

> **Analogía:** en lugar de cortar el libro por número de páginas, se separa **por capítulos**.

Nuestro corpus está en **Markdown** (`#` y `##` marcan los títulos), así que usamos `MarkdownHeaderTextSplitter` de LangChain. Para otros formatos se aplica la misma idea: separar por etiquetas HTML (`HTMLHeaderTextSplitter`), por páginas del PDF o por "Artículo N" en un reglamento.

**Bonus — metadatos:** guardar el título de la sección permite mostrar la **fuente** de la respuesta o filtrar la búsqueda por sección.

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter_md = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "titulo"), ("##", "seccion")],
    strip_headers=False,      # el título queda dentro del texto del fragmento
)
docs = splitter_md.split_text(CORPUS)
docs = [d for d in docs if d.metadata.get("seccion")]   # descartar el bloque que solo tiene el título general

fragmentos = [d.page_content for d in docs]
metadatos  = [{"seccion": d.metadata["seccion"]} for d in docs]

for m, f in zip(metadatos, fragmentos):
    print(f"{m['seccion']:<35} → {len(f)} caracteres")
mostrar_fragmentos(fragmentos, max_frag=3)
estadisticas("6. Estructural", fragmentos)

In [ ]:
ingestar("6. Estructural", fragmentos, metadatos)
evaluar("6. Estructural")

**¿Y si una sección es muy larga?** Se combina con la técnica recursiva: primero se separa por secciones y luego cada sección grande se vuelve a dividir, **conservando el metadato** de la sección:

```python
docs_finales = splitter_rec.split_documents(docs)   # cada fragmento hereda metadata["seccion"]
```

| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Cada fragmento es una unidad temática completa | Requiere que el documento tenga estructura (títulos, etiquetas) |
| Metadatos útiles (sección, página) para citar la fuente | Secciones de tamaños muy distintos |
| Ideal para manuales, reglamentos, documentación técnica | PDFs escaneados o texto plano pierden la estructura |

## 12. Técnica 7 — Semántica

**Idea:** cortar donde **cambia el tema**. Se calcula el embedding de cada oración y se compara con la siguiente: si son muy diferentes (baja similitud), probablemente empieza un tema nuevo y ahí se corta.

> **Analogía:** un lector que va leyendo y pone un separador cada vez que nota que "ahora se habla de otra cosa".

**Algoritmo:**
1. Dividir el texto en oraciones.
2. Calcular el embedding de cada oración (con `task_type="SEMANTIC_SIMILARITY"`, pensado para comparar textos entre sí).
3. Calcular la **distancia** entre oraciones consecutivas: `distancia = 1 − similitud coseno`.
4. Cortar donde la distancia supera un **umbral**; aquí, el **percentil 80** de todas las distancias (el 20 % de saltos más grandes).
5. Como protección, cortar también si el fragmento supera un **tamaño máximo**, para que nunca quede un fragmento enorme.

| Parámetro | Valor |
|---|---|
| `percentil` | 80 (más alto → menos cortes → fragmentos más grandes) |
| `tamano_max` | 1000 caracteres |

> LangChain ofrece esta técnica como `SemanticChunker` (paquete `langchain_experimental`). Aquí la implementamos paso a paso para entender cómo funciona.

In [ ]:
oraciones = dividir_oraciones(CORPUS)
emb_oraciones = np.array(crear_embeddings(oraciones, task_type="SEMANTIC_SIMILARITY"))
emb_oraciones = emb_oraciones / np.linalg.norm(emb_oraciones, axis=1, keepdims=True)

# Distancia entre cada oración y la siguiente
similitudes = np.sum(emb_oraciones[:-1] * emb_oraciones[1:], axis=1)
distancias = 1 - similitudes

PERCENTIL = 80
umbral = np.percentile(distancias, PERCENTIL)
cortes = [i for i, d in enumerate(distancias) if d > umbral]   # cortar DESPUÉS de la oración i
print(f"{len(oraciones)} oraciones | umbral (percentil {PERCENTIL}) = {umbral:.3f} | {len(cortes)} cortes")

**Visualización:** cada punto es el salto de significado entre una oración y la siguiente. Las líneas grises punteadas marcan dónde empieza realmente una **sección** del documento. Si la técnica funciona bien, los picos rojos (cortes) deberían caer **cerca** de esas líneas.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(distancias, marker="o", markersize=3, color="#1F4E79", label="distancia entre oraciones consecutivas")
plt.axhline(umbral, color="orange", linestyle="--", label=f"umbral (percentil {PERCENTIL})")
plt.scatter(cortes, distancias[cortes], color="red", zorder=3, label="corte")
for i, o in enumerate(oraciones[1:]):
    if o.startswith("##"):
        plt.axvline(i, color="gray", linestyle=":", alpha=0.7)
plt.xlabel("Posición (oración i → oración i+1)")
plt.ylabel("Distancia (1 − coseno)")
plt.title("Fragmentación semántica: ¿dónde cambia el tema?")
plt.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def fragmentar_semantico(oraciones, distancias, umbral, tamano_max=1000):
    fragmentos, actual = [], [oraciones[0]]
    for i in range(1, len(oraciones)):
        cambio_de_tema = distancias[i - 1] > umbral
        muy_largo = len(" ".join(actual + [oraciones[i]])) > tamano_max
        if cambio_de_tema or muy_largo:
            fragmentos.append(" ".join(actual))
            actual = []
        actual.append(oraciones[i])
    fragmentos.append(" ".join(actual))
    return fragmentos

fragmentos = fragmentar_semantico(oraciones, distancias, umbral, tamano_max=1000)
mostrar_fragmentos(fragmentos)
estadisticas("7. Semántica", fragmentos)

In [ ]:
ingestar("7. Semántica", fragmentos)
evaluar("7. Semántica")

| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Los fragmentos agrupan ideas relacionadas | Requiere un embedding **por oración** antes de indexar (más costo y tiempo) |
| No depende de que el documento tenga títulos | Tamaños muy variables: puede generar fragmentos enormes o diminutos |
| Útil en textos sin estructura (transcripciones, chats) | El umbral (percentil) hay que ajustarlo por tipo de documento |

## 12-bis. Extra de la actividad: técnica de la semana 10 y técnica propuesta

Para responder las partes 2 y 3 de la actividad se agregan dos técnicas a la comparación, con el mismo corpus, las mismas preguntas y las mismas métricas:

| Técnica | Qué representa |
|---|---|
| **0. Fijo 1000 (semana 10)** | Exactamente el chunking del chatbot `app_llm.ipynb`: `texto[i:i+1000]`, sin solapamiento |
| **8. Recursiva 1000 + 150** | La técnica propuesta: recursiva del mismo tamaño que la semana 10, con 15 % de solapamiento |

In [ ]:
# 0. Técnica usada en la semana 10: tamaño fijo por caracteres, 1000, sin solapamiento
fragmentos = fragmentar_fijo(CORPUS, 1000)
mostrar_fragmentos(fragmentos)
estadisticas("0. Fijo 1000 (semana 10)", fragmentos)
ingestar("0. Fijo 1000 (semana 10)", fragmentos)
evaluar("0. Fijo 1000 (semana 10)")

In [ ]:
# 8. Técnica propuesta: recursiva de 1000 caracteres con 150 de solapamiento
splitter_prop = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""], keep_separator="end")
fragmentos = splitter_prop.split_text(CORPUS)
mostrar_fragmentos(fragmentos)
estadisticas("8. Recursiva 1000 + 150", fragmentos)
ingestar("8. Recursiva 1000 + 150", fragmentos)
evaluar("8. Recursiva 1000 + 150")

## 13. Comparación final

Reunimos las estadísticas y las métricas de las 7 técnicas en una sola tabla.

| Columna | Cómo leerla |
|---|---|
| `fragmentos` | Más fragmentos = más embeddings (costo) |
| `caracteres indexados` | Si supera el tamaño del corpus, hay texto repetido (solapamiento) |
| `% termina en oración` | Qué tanto respeta la técnica los límites naturales del texto |
| `frases clave partidas` | Respuestas que **ninguna** búsqueda podría recuperar completas |
| `Hit@3` / `MRR` | Calidad de la recuperación (más alto = mejor) |

In [ ]:
print(f"Tamaño del corpus: {len(CORPUS):,} caracteres\n")
comparacion = pd.DataFrame(resultados).T
comparacion = comparacion[["fragmentos", "long. media", "caracteres indexados", "% termina en oración",
                           "frases clave partidas", "Hit@3", "MRR"]]
comparacion = comparacion.sort_values(["MRR", "Hit@3"], ascending=False)
comparacion.style.format({"Hit@3": "{:.2f}", "MRR": "{:.2f}"}).background_gradient(subset=["Hit@3", "MRR"], cmap="Greens")

In [ ]:
ax = comparacion[["Hit@3", "MRR"]].plot(kind="barh", figsize=(9, 5), color=["#1F4E79", "#9DC3E6"])
ax.invert_yaxis()
ax.set_xlim(0, 1.05)
ax.set_xlabel("Valor (0 a 1)")
ax.set_title("Calidad de la recuperación por técnica de fragmentación")
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=8)
plt.tight_layout()
plt.show()

> **Ojo con las conclusiones:** usamos un corpus pequeño y solo 7 preguntas, así que una pregunta puede cambiar mucho el resultado. En un proyecto real se evalúa con **decenas o cientos de preguntas** sobre los documentos reales. Lo valioso aquí es el **método**: definir preguntas, medir y comparar antes de elegir.

## 14. RAG lado a lado: ¿cómo afecta la fragmentación a la respuesta final?

Tomamos una pregunta y generamos la respuesta con `gemini-2.5-flash` usando los fragmentos recuperados por la técnica **con peor MRR** y por la técnica **con mejor MRR**. El prompt es el mismo que el del chatbot.

In [ ]:
def generar_respuesta(pregunta, contextos):
    contexto = "\n\n".join(c["texto"] for c in contextos)
    prompt = f'''Eres un asistente experto. Usa EXCLUSIVAMENTE el siguiente contexto para responder la pregunta del usuario. Si la respuesta no está en el contexto, indícalo claramente.

Contexto:
{contexto}

Pregunta: {pregunta}

Responde de forma concisa y clara en español.'''
    return client_genai.models.generate_content(model=MODELO_LLM, contents=prompt).text

pregunta = PREGUNTAS[3]["pregunta"]          # puedes cambiar el índice (0 a 6)
peor, mejor = comparacion.index[-1], comparacion.index[0]

for tecnica in [peor, mejor]:
    contextos = buscar(pregunta, tecnica, k=3)
    print("=" * 90)
    print(f"Técnica: {tecnica}   (MRR = {resultados[tecnica]['MRR']:.2f})")
    print(f"Pregunta: {pregunta}\n")
    for i, c in enumerate(contextos, 1):
        print(f"  [{i}] score={c['score']:.3f} | {c['texto'][:120].replace(chr(10), ' ')}...")
    print("\nRespuesta:", generar_respuesta(pregunta, contextos))

## 15. Resumen: ¿qué técnica usar según el caso?

### 15.1 Resumen de las técnicas

| # | Técnica | Cómo corta | Ventaja principal | Desventaja principal | Costo |
|---|---|---|---|---|---|
| 1 | Fijo (caracteres) | Cada *N* caracteres | Simplicidad | Corta palabras e ideas | Bajo |
| 2 | Fijo + solapamiento | Cada *N* caracteres, repitiendo un margen | Menos pérdida en los bordes | Texto duplicado | Bajo–medio |
| 3 | Por tokens | Cada *N* tokens | Controla el límite del modelo | Depende del tokenizador | Bajo |
| 4 | Por oraciones | Grupos de *N* oraciones | No corta oraciones | Tamaño variable | Bajo |
| 5 | Recursiva | Párrafo → línea → oración → palabra | Buen equilibrio general | Ajustar parámetros | Bajo |
| 6 | Estructural | Títulos, secciones, páginas | Fragmentos temáticos + metadatos | Requiere estructura | Bajo |
| 7 | Semántica | Donde cambia el significado | Agrupa ideas relacionadas | Embedding por oración | Alto |

### 15.2 Guía de decisión por tipo de caso

| Caso | Técnica recomendada | ¿Por qué? |
|---|---|---|
| Prototipo rápido o prueba de concepto | **Recursiva** (500–1000 car., 10–20 % solapamiento) | Buen resultado sin esfuerzo |
| Manuales, documentación técnica, páginas web con títulos | **Estructural** (+ recursiva si hay secciones largas) | Cada sección es un tema; los metadatos permiten citar la fuente |
| Reglamentos, contratos, normas legales | **Estructural** por artículo o cláusula | La unidad de consulta es el artículo |
| Preguntas frecuentes (FAQ) | **Estructural**: una pregunta + su respuesta por fragmento | Cada par pregunta-respuesta es independiente |
| Transcripciones de reuniones, chats, entrevistas | **Semántica** o **por oraciones** | No hay títulos; los temas cambian sin aviso |
| Textos narrativos (noticias, artículos) | **Por oraciones** o **recursiva** | Respeta el flujo de las oraciones |
| Fragmentos cerca del límite del modelo o control estricto de costo | **Por tokens** | Se mide en la misma unidad que el modelo |
| Logs o datos con formato fijo (registros, CSV) | **Fijo** (por registro o por líneas) | La estructura ya es regular |
| PDFs escaneados o texto "sucio" | **Recursiva** con solapamiento (tras OCR y limpieza) | Robusta ante estructura pobre |

### 15.3 Recomendaciones prácticas
1. **Empieza simple:** recursiva con solapamiento. Solo cambia de técnica si las métricas lo justifican.
2. **Mide siempre:** arma un conjunto de preguntas con respuesta conocida y calcula Hit@k y MRR antes de elegir.
3. **El tamaño importa tanto como la técnica:** prueba 2 o 3 tamaños (p. ej. 300, 500, 1000).
4. **Guarda metadatos** (sección, página, fuente): mejoran la trazabilidad y permiten filtrar.
5. **No existe una técnica universal:** depende del documento y del tipo de pregunta.

## 16. Limpieza
Borra los documentos del laboratorio para no ocupar espacio en el clúster gratuito (M0). Si quieres eliminar también el índice, descomenta la última línea.

In [ ]:
borrados = collection.delete_many({}).deleted_count
print(f"{borrados} documentos eliminados")
# collection.drop_search_index(INDICE)